In [ ]:
"""Offline evaluation for recommendation algorithms.

This notebook evaluates different recommendation methods (hybrid, als, session, vector)
using historical interaction data. It assumes the backend is configured and the
same adapters/models are available as in the API.
"""

import os
import sys
from typing import List, Dict, Any

import numpy as np
from loguru import logger

# Ensure project root on path
ROOT_DIR = os.path.dirname(os.path.dirname(os.path.abspath("__file__")))
if ROOT_DIR not in sys.path:
    sys.path.append(ROOT_DIR)

from adapters.factory import get_user_behavior
from models.recommendations import get_product_recommender


# ----------------------------
# Configuration
# ----------------------------

METHODS = ["hybrid", "als", "session", "vector"]
K = 10  # top-K to evaluate
MAX_USERS = 500  # limit users for quick evaluation
MIN_HISTORY = 3  # min interactions required to create a query


# ----------------------------
# Helper functions
# ----------------------------

behavior = get_user_behavior()
recommender = get_product_recommender()


def _get_unique_users(limit: int) -> List[str]:
    """Return list of user_ids with at least MIN_HISTORY interactions."""
    interactions = behavior.get_interaction_counts(limit=limit * 10)
    by_user: Dict[str, int] = {}
    for row in interactions:
        uid = str(row.get("user_id"))
        by_user[uid] = by_user.get(uid, 0) + int(row.get("count", 0) or 0)
    users = [u for u, c in by_user.items() if c >= MIN_HISTORY]
    return users[:limit]


def _split_history(user_id: str) -> Dict[str, Any]:
    """Split a user's history into context (input) and targets (future items).

    Very simple split: use all but last interaction as context, last item as target.
    """
    hist = behavior.get_user_history(user_id, limit=50)
    if len(hist) < MIN_HISTORY:
        return {"context": [], "targets": []}

    # Sorted by timestamp descending already; reverse to chronological
    hist = list(reversed(hist))
    context = hist[:-1]
    targets = hist[-1:]
    return {"context": context, "targets": targets}


def _get_recs(user_id: str, method: str, k: int) -> List[str]:
    """Call recommender for a given method and return list of product_ids."""
    method = method.lower().strip()
    if method == "vector":
        recs = recommender.get_personalized_recommendations(user_id=user_id, limit=k)
    elif method == "als":
        recs = recommender.get_als_recommendations(user_id=user_id, limit=k, train_if_missing=True)
    elif method == "session":
        recs = recommender.get_session_based_recommendations(user_id=user_id, limit=k)
    elif method == "hybrid":
        recs = recommender.get_hybrid_recommendations(user_id=user_id, limit=k)
    else:
        raise ValueError(f"Unknown method: {method}")

    return [str(r.get("product_id")) for r in recs][:k]


def _hits_at_k(recs: List[str], targets: List[str]) -> float:
    if not recs or not targets:
        return 0.0
    return 1.0 if any(t in recs for t in targets) else 0.0


# ----------------------------
# Main evaluation loop
# ----------------------------

users = _get_unique_users(limit=MAX_USERS)
logger.info(f"Evaluating on {len(users)} users")

results = {m: [] for m in METHODS}

for uid in users:
    split = _split_history(uid)
    ctx, tgt = split["context"], split["targets"]
    if not ctx or not tgt:
        continue
    target_ids = [str(t.get("product_id")) for t in tgt if t.get("product_id")]
    if not target_ids:
        continue

    for m in METHODS:
        try:
            rec_ids = _get_recs(uid, m, K)
            hit = _hits_at_k(rec_ids, target_ids)
            results[m].append(hit)
        except Exception as e:
            logger.warning(f"Eval error for user={uid}, method={m}: {e}")


# Aggregate metrics
summary = {}
for m, hits in results.items():
    if hits:
        hr = float(np.mean(hits))
        summary[m] = {"HitRate@K": hr, "num_users": len(hits)}
    else:
        summary[m] = {"HitRate@K": 0.0, "num_users": 0}

logger.info("Offline evaluation summary:")
for m, s in summary.items():
    logger.info(f"  {m}: HitRate@{K}={s['HitRate@K']:.4f} over {s['num_users']} users")

summary
